# 目标管理系统 (Goal Management System)

## 学习目标

本教程将帮助你理解自主智能体中的目标管理系统，包括：

1. **Goal 数据结构** - 目标的表示与状态管理
2. **目标分解 (HTN)** - 将复杂目标分解为子任务
3. **优先级队列** - 基于堆的目标调度
4. **完成检测** - 判断目标是否达成

---

In [ ]:
# 环境设置
import sys
sys.path.insert(0, '../src')

from goal_manager import (
    Goal, GoalStatus, GoalPriority,
    GoalManager, GoalPriorityQueue,
    LLMGoalDecomposer, RuleBasedDecomposer,
    CompletionChecker
)

## 1. Goal 数据结构

### 1.1 创建目标

每个目标包含描述、优先级、状态等属性：

In [ ]:
# 创建一个简单目标
goal = Goal(
    description="编写一个 Python Web 应用",
    priority=GoalPriority.HIGH,
    success_criteria="应用能够处理 HTTP 请求并返回响应"
)

print(f"目标 ID: {goal.goal_id}")
print(f"描述: {goal.description}")
print(f"优先级: {goal.priority.name}")
print(f"状态: {goal.status.name}")
print(f"成功标准: {goal.success_criteria}")

### 1.2 目标状态转换

目标有四种状态：PENDING → IN_PROGRESS → COMPLETED/FAILED

In [ ]:
# 状态转换演示
goal = Goal(description="测试目标")
print(f"初始状态: {goal.status.name}")

# 开始执行
goal.mark_in_progress()
print(f"开始后: {goal.status.name}")

# 完成目标
goal.mark_completed()
print(f"完成后: {goal.status.name}")
print(f"完成时间: {goal.completed_at}")

### 1.3 重试机制

目标失败后可以重试，直到达到最大尝试次数：

In [ ]:
# 重试机制演示
goal = Goal(description="可能失败的任务", max_attempts=3)

for i in range(4):
    goal.mark_in_progress()
    goal.mark_failed(can_retry=True)
    print(f"尝试 {i+1}: 状态={goal.status.name}, 尝试次数={goal.attempt_count}")

## 2. 目标分解 (Hierarchical Task Network)

### 2.1 基于规则的分解

根据目标描述中的关键词自动分解：

In [ ]:
# 规则分解器
decomposer = RuleBasedDecomposer()

# 分解 "写" 类型的目标
write_goal = Goal(description="写一份技术报告")
sub_goals = decomposer.decompose(write_goal)

print(f"原目标: {write_goal.description}")
print(f"\n分解为 {len(sub_goals)} 个子目标:")
for i, sg in enumerate(sub_goals, 1):
    print(f"  {i}. {sg.description}")

In [ ]:
# 分解 "构建" 类型的目标
build_goal = Goal(description="构建一个聊天机器人")
sub_goals = decomposer.decompose(build_goal)

print(f"原目标: {build_goal.description}")
print(f"\n分解为 {len(sub_goals)} 个子目标:")
for i, sg in enumerate(sub_goals, 1):
    print(f"  {i}. {sg.description}")

### 2.2 LLM 驱动的分解

使用大语言模型进行更智能的分解（需要提供 LLM 函数）：

In [ ]:
# 模拟 LLM 函数
def mock_llm(prompt: str) -> str:
    """模拟 LLM 响应"""
    return """1. 设计数据库模型
2. 实现用户认证 API
3. 创建前端界面
4. 编写单元测试
5. 部署到服务器"""

# LLM 分解器
llm_decomposer = LLMGoalDecomposer(llm_func=mock_llm)

complex_goal = Goal(description="开发一个用户管理系统")
sub_goals = llm_decomposer.decompose(complex_goal)

print(f"原目标: {complex_goal.description}")
print(f"\nLLM 分解为 {len(sub_goals)} 个子目标:")
for i, sg in enumerate(sub_goals, 1):
    print(f"  {i}. {sg.description} (优先级: {sg.priority.name})")

## 3. 优先级队列

### 3.1 基本操作

使用堆实现的优先级队列，高优先级目标先出队：

In [ ]:
# 创建优先级队列
queue = GoalPriorityQueue()

# 添加不同优先级的目标
goals = [
    Goal("低优先级任务", priority=GoalPriority.LOW),
    Goal("紧急任务", priority=GoalPriority.CRITICAL),
    Goal("普通任务", priority=GoalPriority.MEDIUM),
    Goal("重要任务", priority=GoalPriority.HIGH),
]

for g in goals:
    queue.push(g)
    print(f"添加: {g.description} ({g.priority.name})")

print(f"\n队列大小: {len(queue)}")

In [ ]:
# 按优先级顺序出队
print("出队顺序 (高优先级先出):")
while len(queue) > 0:
    g = queue.pop()
    print(f"  {g.priority.name}: {g.description}")

### 3.2 查看和移除

支持 peek（查看但不移除）和 remove（按 ID 移除）操作：

In [ ]:
# 重新填充队列
queue = GoalPriorityQueue()
g1 = Goal("任务 A", priority=GoalPriority.HIGH)
g2 = Goal("任务 B", priority=GoalPriority.MEDIUM)
queue.push(g1)
queue.push(g2)

# Peek 操作
top = queue.peek()
print(f"队首 (peek): {top.description}")
print(f"队列大小: {len(queue)} (peek 不改变大小)")

# Remove 操作
removed = queue.remove(g2.goal_id)
print(f"\n移除任务 B: {removed}")
print(f"队列大小: {len(queue)}")

## 4. 完成检测

### 4.1 关键词检测

通过结果中的关键词判断目标是否完成：

In [ ]:
checker = CompletionChecker()

goal = Goal(description="完成数据分析")

# 测试不同的结果
results = [
    "任务已完成，生成了分析报告",
    "分析成功，发现了 3 个关键趋势",
    "正在处理数据...",
    "遇到错误，无法继续",
]

for result in results:
    is_done, reason = checker.check_completion(goal, result)
    status = "✓ 完成" if is_done else "✗ 未完成"
    print(f"{status}: \"{result[:30]}...\"")
    if is_done:
        print(f"   原因: {reason}")

## 5. GoalManager 综合使用

### 5.1 完整工作流

In [ ]:
# 创建目标管理器
manager = GoalManager()

# 添加主目标
main_goal = manager.add_goal(
    "开发一个 TODO 应用",
    priority=GoalPriority.HIGH
)
print(f"添加主目标: {main_goal.description}")

# 分解目标
sub_goals = manager.decompose_goal(main_goal.goal_id)
print(f"\n分解为 {len(sub_goals)} 个子目标:")
for sg in sub_goals:
    print(f"  - {sg.description}")

In [ ]:
# 执行目标
print("开始执行目标...\n")

for i in range(3):
    # 获取下一个目标
    next_goal = manager.get_next_goal()
    if not next_goal:
        print("没有更多目标")
        break
    
    print(f"执行: {next_goal.description}")
    
    # 开始执行
    manager.start_goal(next_goal.goal_id)
    
    # 模拟执行结果
    result = f"{next_goal.description} 已完成"
    is_done, reason = manager.complete_goal(next_goal.goal_id, result)
    
    print(f"  结果: {'完成' if is_done else '未完成'}")
    print()

In [ ]:
# 查看进度
progress = manager.get_progress()
print("目标进度:")
print(f"  总数: {progress['total']}")
print(f"  已完成: {progress['completed']}")
print(f"  进行中: {progress['in_progress']}")
print(f"  待处理: {progress['pending']}")
print(f"  完成率: {progress['completion_rate']:.1%}")

## 6. 练习

### 练习 1: 创建目标层次结构

创建一个三层的目标结构：主目标 → 子目标 → 子子目标

In [ ]:
# 你的代码
# manager = GoalManager()
# ...

### 练习 2: 实现自定义分解器

创建一个针对特定领域（如数据科学）的目标分解器

In [ ]:
# 你的代码
# class DataScienceDecomposer:
#     def decompose(self, goal):
#         ...

## 总结

本教程介绍了目标管理系统的核心组件：

| 组件 | 功能 | 关键方法 |
|:-----|:-----|:---------|
| `Goal` | 目标数据结构 | `mark_completed()`, `mark_failed()` |
| `GoalDecomposer` | 目标分解 | `decompose()` |
| `GoalPriorityQueue` | 优先级调度 | `push()`, `pop()`, `peek()` |
| `CompletionChecker` | 完成检测 | `check_completion()` |
| `GoalManager` | 综合管理 | `add_goal()`, `get_next_goal()` |

下一教程将介绍动作执行系统。